# Agentic D&D Manager

In [5]:
!pip3 install haystack

In [6]:
import os
import json
from typing import Annotated, Optional

# Data handling
import pandas as pd  # Fast dataframe library (alternative: pandas)

# Haystack agent framework
from haystack import Pipeline
from haystack.components.agents import Agent
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.dataclasses import ChatMessage
from haystack.tools import create_tool_from_function
from haystack_experimental.chat_message_stores.in_memory import InMemoryChatMessageStore
from haystack_experimental.components.retrievers import ChatMessageRetriever
from haystack_experimental.components.writers import ChatMessageWriter

ModuleNotFoundError: No module named 'haystack'

In [ ]:
# ============ LOAD DATA ============
def load_session_data(filepath="session_data.csv"):
    """Load session data. If file doesn't exist, create empty tables."""
    if os.path.exists(filepath):
        # Load session data from CSV (we'll use a pickle file for the dict of dataframes)
        import pickle
        with open(filepath.replace('.csv', '.pkl'), 'rb') as f:
            data = pickle.load(f)
        return data
    else:
        # If starting fresh, create empty schema
        return initialize_empty_session()


def initialize_empty_session():
    """Create fresh empty tables for a new session."""
    return {
        "players": pd.DataFrame({
            "name": [],
            "health": [],
            "defense": [],
            "damage": [],
            "origin": [],
        }),
        "inventory": pd.DataFrame({
            "player_name": [],
            "item_name": [],
            "description": [],
            "quantity": [],
            "is_consumable": [],
            "is_equippable": [],
        }),
        "status_effects": pd.DataFrame({
            "player_name": [],
            "effect": [],
            "duration": [],  # in turns
            "details": [],
        }),
        "skills": pd.DataFrame({
            "player_name": [],
            "skill_name": [],
            "details": [],
            "cooldown": [],  # in turns
            "cost": [],  # e.g., mana, stamina
        }),
        "agent_notes": pd.DataFrame({
            "timestamp": [],
            "event": [],
            "notes": [],
        }),
    }


# ============ EXAMPLE: Load & Access ============
session_data = load_session_data()

# Query: Get "Zero" player
player_zero = session_data["players"][session_data["players"]["name"] == "zero"]
# Returns:
# name health defense damage origin
# 0 zero 20 5 6 lab

# Query: Get all items for "Zero"
zero_inventory = session_data["inventory"][session_data["inventory"]["player_name"] == "zero"]


# ============ MODIFY DATA ============
def add_item_to_player(session_data, player_name, item_name, description, quantity, is_consumable, is_equippable):
    """Add an item to a player's inventory (or increase quantity if exists)."""
    
    inv = session_data["inventory"]
    
    # Check if this item already exists for this player
    existing = inv[(inv["player_name"] == player_name) & (inv["item_name"] == item_name)]
    
    if len(existing) > 0:
        # Item exists, increase quantity
        idx = existing.index[0]
        session_data["inventory"].loc[idx, "quantity"] += quantity
    else:
        # New item, append to inventory
        new_row = pd.DataFrame({
            "player_name": [player_name],
            "item_name": [item_name],
            "description": [description],
            "quantity": [quantity],
            "is_consumable": [is_consumable],
            "is_equippable": [is_equippable],
        })
        session_data["inventory"] = pd.concat([inv, new_row], ignore_index=True)
    
    return session_data


# ============ SAVE DATA ============
def save_session_data(session_data, filepath="session_data.pkl"):
    """Save all tables to a pickle file (preserves the dict structure)."""
    import pickle
    with open(filepath, 'wb') as f:
        pickle.dump(session_data, f)
    print(f"Session data saved to {filepath}")


# ============ WORKFLOW ============
# At the start of your session:
session_data = load_session_data()

# ... agent makes changes ...
session_data = add_item_to_player(session_data, "zero", "Potion", "Heals 5 HP", 1, True, False)

# At the end:
save_session_data(session_data)

# Next time you run, it loads the updated data

In [ ]:
def get_llm(use_local=False):
    """
    Get an LLM instance. 
    - use_local=True: Use a locally-downloaded model (e.g., via Ollama)
    - use_local=False: Use remote API
    """
    
    if use_local:
        # For local models, you'd need something like:
        # ollama pull mistral  # (run this in terminal first)
        # Then in Python:
        from ollama import generate
        
        # Wrapper to make local model compatible with Haystack
        class LocalLLMWrapper:
            def __init__(self, model_name="mistral"):
                self.model_name = model_name
            
            def generate(self, messages, **kwargs):
                # Format messages for local model
                prompt = "\n".join([f"{m['role']}: {m['content']}" for m in messages])
                result = generate(model=self.model_name, prompt=prompt, stream=False)
                return {"response": result}
        
        llm = LocalLLMWrapper(model_name="mistral")
        print("Using local Mistral model")
    else:
        # Remote API
        BASE_URL = "https://sdsu-rci-owui.nrp-nautilus.io/api"
        API_KEY = open("./api_key.txt").read().strip()
        ANSWER_MODEL = "gpt-oss:120b"
        
        os.environ["OPENAI_API_KEY"] = API_KEY
        
        llm = OpenAIChatGenerator(
            model=ANSWER_MODEL,
            api_base_url=BASE_URL,
            generation_kwargs={"temperature": 0.5, "reasoning_effort": "high"},
        )
        print("Using remote API model")
    
    return llm


# Use it:
llm = get_llm(use_local=False)  # Change to True if you want local
